In [ ]:
import json
import random
import time
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from pathlib import Path

# ==================== CẤU HÌNH ====================
MODEL_ID = "Qwen/Qwen3-4B"
ADAPTER_DIR = "/kaggle/input/qwen-finetuned/transformers/default/3/qwen_finetuned"
EVAL_JSONL = Path("/kaggle/input/evaluate-classification/evaluation_questions.jsonl")
SHUFFLE_SEED = 42  # đổi hoặc set None để không cố định seed
MAX_NEW_TOKENS = 12

# ==================== LOAD TOKENIZER + MODEL ====================
print(" Đang tải tokenizer và model... (có thể mất vài phút)")
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR)
# đảm bảo có pad_token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True
)

model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
model.eval()
# get device of first parameter (works with device_map="auto")
_device = next(model.parameters()).device
print(f" Model và tokenizer đã tải xong. Thiết bị chính: {_device}")

# ==================== HÀM HỖ TRỢ ====================
def normalize_label(label: str) -> str:
    """Chuẩn hoá nhãn từ file (khác nhau về case/format) -> one of HISTORY_DIRECT, OUT_OF_DOMAIN, INSUFFICIENT_INFO, VAGUE"""
    if not isinstance(label, str):
        return "HISTORY_DIRECT"
    s = label.strip().lower().replace(" ", "_")
    mapping = {
        "history_direct": "HISTORY_DIRECT",
        "out_of_domain": "OUT_OF_DOMAIN",
        "insufficient_info": "INSUFFICIENT_INFO",
        "vague": "VAGUE",
    }
    return mapping.get(s, s.upper())

# ==================== CLASSIFIER ====================
class QuestionClassifier:
    def __init__(self, tokenizer, model, device):
        self.tokenizer = tokenizer
        self.model = model
        self.device = device
    def classify_question(self, question: str) -> str:
        """Phân loại câu hỏi với prompt cải tiến và optional self-consistency"""
        
        classification_prompt = f"""PHÂN LOẠI CÂU HỎI: Chọn MỘT trong 4 loại dưới đây:
            
            1. HISTORY_DIRECT: Câu hỏi CỤ THỂ về sự kiện, nhân vật, thời gian lịch sử Việt Nam (có chi tiết rõ ràng)
            2. OUT_OF_DOMAIN: Câu hỏi HOÀN TOÀN KHÔNG LIÊN QUAN đến lịch sử/văn hóa Việt Nam (thời tiết, thể thao, ẩm thực, công nghệ hiện đại)
            3. INSUFFICIENT_INFO: Câu hỏi về lịch sử Việt Nam nhưng THÔNG TIN KHÔNG TỒN TẠI (ví dụ: công nghệ hiện đại trong thời phong kiến, phát minh không có thật)
            4. VAGUE: Câu hỏi QUÁ RỘNG, MƠ HỒ, THIẾU CHI TIẾT (cần làm rõ)
            
            QUAN TRỌNG: 
            - INSUFFICIENT_INFO: vẫn là câu hỏi về lịch sử, nhưng thông tin không có thật
            - OUT_OF_DOMAIN: không phải câu hỏi về lịch sử
            
            VÍ DỤ:
            - "Vua Quang Trung đánh quân Thanh năm nào?" -> HISTORY_DIRECT  
            - "Thời tiết Hà Nội thế nào?" -> OUT_OF_DOMAIN
            - "Nhà Trần có dùng điện thoại không?" -> INSUFFICIENT_INFO (về lịch sử nhưng không có thật)
            - "Triều Nguyễn có Internet không?" -> INSUFFICIENT_INFO  
            - "Kể về lịch sử Việt Nam" -> VAGUE
            - "Cristiano Ronaldo là ai?" -> OUT_OF_DOMAIN
            
            CÂU HỎI CẦN PHÂN LOẠI: {question}
            
            CHỈ TRẢ LỜI MỘT TỪ: HISTORY_DIRECT, OUT_OF_DOMAIN, INSUFFICIENT_INFO, hoặc VAGUE
            Kết quả:"""

        # Tokenize and move to correct device
        inputs = self.tokenizer(classification_prompt, return_tensors="pt", truncation=True, max_length=1024)
        inputs = {k: v.to(self.device) for k, v in inputs.items()}

        with torch.inference_mode():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                temperature=0.1,
                pad_token_id=self.tokenizer.eos_token_id,
                eos_token_id=self.tokenizer.eos_token_id
            )

        # Decode and extract predicted label
        resp = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
        # Nếu model echo prompt, cắt prompt phía trước
        if resp.startswith(classification_prompt):
            classification = resp[len(classification_prompt):].strip()
        else:
            # Nếu không, lấy phần cuối sau dòng "Kết quả:" nếu có
            if "Kết quả:" in resp:
                classification = resp.split("Kết quả:")[-1].strip()
            else:
                classification = resp.strip()

        # Lấy token đầu tiên (một từ) và chuẩn hoá
        first_token = classification.split()[0] if classification else ""
        candidate = first_token.upper().strip().strip('":,.')
        # Bảo đảm khớp với 4 nhãn
        if "HISTORY" in candidate:
            return "HISTORY_DIRECT"
        if "OUT" in candidate or "DOMAIN" in candidate:
            return "OUT_OF_DOMAIN"
        if "INSUFFICIENT" in candidate or "INSUFFICIENT_INFO" in candidate:
            return "INSUFFICIENT_INFO"
        if "VAGUE" in candidate:
            return "VAGUE"

        # Nếu không chắc, thử các từ trong toàn chuỗi classification
        c_up = classification.upper()
        if "HISTORY_DIRECT" in c_up:
            return "HISTORY_DIRECT"
        if "OUT_OF_DOMAIN" in c_up or "OUT-OF-DOMAIN" in c_up:
            return "OUT_OF_DOMAIN"
        if "INSUFFICIENT" in c_up:
            return "INSUFFICIENT_INFO"
        if "VAGUE" in c_up:
            return "VAGUE"

        # Mặc định fallback
        print(f" Không phân loại rõ: '{classification}' -> fallback HISTORY_DIRECT")
        return "HISTORY_DIRECT"

# ==================== ĐỌC FILE JSONL ====================
def load_eval_questions(path: Path, shuffle: bool = True):
    if not path.exists():
        raise FileNotFoundError(f"Không tìm thấy file: {path}")
    questions = []
    with path.open("r", encoding="utf-8") as f:
        for i, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
                q = obj.get("question") or obj.get("text") or obj.get("q") or ""
                t = obj.get("type") or obj.get("label") or ""
                if q == "":
                    continue
                questions.append({"question": q, "type": normalize_label(t)})
            except json.JSONDecodeError:
                print(f" Bỏ dòng {i}: không phải JSON hợp lệ.")
    if shuffle:
        if SHUFFLE_SEED is not None:
            random.Random(SHUFFLE_SEED).shuffle(questions)
        else:
            random.shuffle(questions)
    return questions

# ==================== CHẠY ĐÁNH GIÁ ====================
def run_classification_tests_from_file(jsonl_path: Path):
    try:
        test_items = load_eval_questions(jsonl_path, shuffle=True)
    except FileNotFoundError as e:
        print(str(e))
        return [], 0.0

    print(f" BẮT ĐẦU ĐÁNH GIÁ TỪ FILE: {jsonl_path} (tổng {len(test_items)} câu hỏi)")
    classifier = QuestionClassifier(tokenizer, model, _device)

    results = []
    for i, item in enumerate(test_items, 1):
        question = item["question"]
        expected = item["type"]
        start = time.time()
        predicted = classifier.classify_question(question)
        elapsed = time.time() - start
        is_correct = predicted == expected
        status = "✅" if is_correct else "❌"
        print(f"{i:03d}. {status} Expected={expected:18} Predicted={predicted:18} Time={elapsed:.2f}s")
        if not is_correct:
            print(f"     Q: {question}")
        results.append({
            "question": question,
            "expected": expected,
            "predicted": predicted,
            "correct": is_correct,
            "time": elapsed
        })

    # Tổng hợp kết quả
    total = len(results)
    correct = sum(1 for r in results if r["correct"])
    accuracy = correct / total if total > 0 else 0.0
    print("\n" + "="*60)
    print(" KẾT QUẢ TỔNG HỢP")
    print(f"Tổng số test: {total}")
    print(f"Số test đúng: {correct}")
    print(f"Độ chính xác: {accuracy:.2%}")

    # Phân tích theo loại
    type_stats = {}
    for r in results:
        t = r["expected"]
        if t not in type_stats:
            type_stats[t] = {"total": 0, "correct": 0}
        type_stats[t]["total"] += 1
        if r["correct"]:
            type_stats[t]["correct"] += 1

    print("\n PHÂN TÍCH THEO LOẠI:")
    for t, s in type_stats.items():
        acc = s["correct"] / s["total"] if s["total"] else 0.0
        print(f"  {t}: {s['correct']}/{s['total']} ({acc:.2%})")

    # Liệt kê một số sai lệch
    wrongs = [r for r in results if not r["correct"]]
    if wrongs:
        print(f"\n Tổng {len(wrongs)} trường hợp sai — hiển thị tối đa 20:")
        for r in wrongs[:20]:
            print(f" - Expected={r['expected']:18} Predicted={r['predicted']:18} Q: {r['question']}")

    return results, accuracy

# ==================== MAIN ====================
if __name__ == "__main__":
    # Nếu file không tồn tại, báo lỗi rõ ràng
    if not EVAL_JSONL.exists():
        print(" File đánh giá không tồn tại tại đường dẫn:", EVAL_JSONL)
        print("Hãy kiểm tra đường dẫn hoặc tải file vào /kaggle/input/classification-data/")
    else:
        results, acc = run_classification_tests_from_file(EVAL_JSONL)


